Realização em espaço de estados a partir de uma Função de Transferência $G(s)$ ou obtém as matrizes da Forma Canônica Controlável (FCC) e Forma Canônica Observável (FCO).

In [1]:
import numpy as np
import sympy as sp
import control as ct

def obter_polinomio_usuario(prompt):
    """Lê coeficientes informados pelo usuário e retorna uma lista de floats."""
    while True:

        try:
            entrada = input(prompt)
            coefs = [float(x.strip()) for x in entrada.split(',')]
            return coefs
        except ValueError:
            print("Entrada inválida! Insira os coeficientes separados por vírgula. Ex: 1, 2, 3")

def exibir_matriz(nome, matriz):
    """Exibe matrizes formatadas na tela."""
    print(f"\nMatriz {nome}:")
    if isinstance(matriz, np.ndarray):
        print(np.array2string(matriz, precision=4, suppress_small=True))
    else:
        sp.pprint(matriz)

def forma_canonica_controlavel(num_coefs, den_coefs):
    """
    Gera a Forma Canônica Controlável (FCC) para um sistema estritamente próprio.
    Num: b_m*s^m + ... + b_0
    Den: s^n + a_{n-1}*s^{n-1} + ... + a_0
    """
    # Normaliza o denominador para que o coeficiente líder seja 1
    a_n = den_coefs[0]
    den = [x / a_n for x in den_coefs]
    num = [x / a_n for x in num_coefs]
    
    n = len(den) - 1 # Ordem do sistema
    
    # Ajusta o numerador para ter tamanho (n + 1) preenchendo com zeros à esquerda
    num_pad = [0.0] * (len(den) - len(num)) + num
    
    d = num_pad[0] # Termo direto (D)
    b_tilde = [num_pad[i] - d * den[i] for i in range(1, n + 1)]
    
    # Matriz A na FCC (Forma de Companion Superior/Inferior)
    A = np.zeros((n, n))
    for i in range(n - 1):
        A[i, i + 1] = 1.0
    A[-1, :] = -np.array(den[1:][::-1])
    
    # Matriz B na FCC
    B = np.zeros((n, 1))
    B[-1, 0] = 1.0
    
    # Matriz C na FCC
    C = np.array([b_tilde[::-1]])
    
    # Matriz D
    D = np.array([[d]])
    
    return A, B, C, D

def analisar_sistemas(A, B, C, D):
    """Verifica Controlabilidade, Observabilidade e Polinômio Característico."""
    sys = ct.StateSpace(A, B, C, D)
    
    # Controlabilidade
    Ctr = ct.ctrb(A, B)
    posto_ctr = np.linalg.matrix_rank(Ctr)
    e_controlavel = (posto_ctr == A.shape[0])
    
    # Observabilidade
    Obs = ct.obsv(A, C)
    posto_obs = np.linalg.matrix_rank(Obs)
    e_observavel = (posto_obs == A.shape[0])
    
    # Autovalores
    autovalores = np.linalg.eigvals(A)
    
    print("\n" + "="*50)
    print("ANÁLISE DE PROPRIEDADES DO ESPAÇO DE ESTADOS")
    print("="*50)
    print(f"Ordem do sistema (n): {A.shape[0]}")
    print(f"Posto da Matriz de Controlabilidade: {posto_ctr} -> {'CONTROLÁVEL' if e_controlavel else 'NÃO CONTROLÁVEL'}")
    print(f"Posto da Matriz de Observabilidade:  {posto_obs} -> {'OBSERVÁVEL' if e_observavel else 'NÃO OBSERVÁVEL'}")
    print("\nAutovalores de A (Pólos do Sistema):")
    for idx, lamb in enumerate(autovalores, 1):
        print(f"  λ{idx} = {lamb:.4f}")

def main():
    print("="*60)
    print(" REALIZAÇÃO EM ESPAÇO DE ESTADOS - SCRIPT INTERATIVO")
    print("="*60)
    print("Defina a Função de Transferência G(s) = N(s) / D(s)\n")
    
    # Entrada de Dados
    num = obter_polinomio_usuario("Digite os coeficientes do NUMERADOR (ex: 2, 3 para 2s + 3): ")
    den = obter_polinomio_usuario("Digite os coeficientes do DENOMINADOR (ex: 1, 5, 6 para s^2 + 5s + 6): ")
    
    if len(num) > len(den):
        print("\n[ERRO] O grau do numerador não pode ser maior que o do denominador (sistema impróprio).")
        return
        
    # Exibe a FT em formato simbólico
    s = sp.Symbol('s')
    num_poly = sum(c * s**(len(num)-1-i) for i, c in enumerate(num))
    den_poly = sum(c * s**(len(den)-1-i) for i, c in enumerate(den))
    G_sym = num_poly / den_poly
    
    print("\nFunção de Transferência G(s):")
    sp.pprint(G_sym)
    
    # Seleção de Forma Canônica
    print("\nEscolha o formato de realização:")
    print("1 - Forma Canônica Controlável (FCC)")
    print("2 - Forma Canônica Observável (FCO)")
    
    opcao = input("\nOpção (1 ou 2) [Padrão: 1]: ").strip()
    
    A, B, C, D = forma_canonica_controlavel(num, den)
    
    if opcao == '2':
        # FCO é a transposta da FCC: A_fco = A_fcc^T, B_fco = C_fcc^T, C_fco = B_fcc^T
        A = A.T
        B_temp = B.copy()
        B = C.T
        C = B_temp.T
        tipo = "Forma Canônica Observável (FCO)"
    else:
        tipo = "Forma Canônica Controlável (FCC)"
        
    print("\n" + "="*50)
    print(f"REALIZAÇÃO: {tipo}")
    print("="*50)
    
    exibir_matriz("A", A)
    exibir_matriz("B", B)
    exibir_matriz("C", C)
    exibir_matriz("D", D)
    
    # Análise de Propriedades
    analisar_sistemas(A, B, C, D)

if __name__ == "__main__":
    main()

 REALIZAÇÃO EM ESPAÇO DE ESTADOS - SCRIPT INTERATIVO
Defina a Função de Transferência G(s) = N(s) / D(s)


Função de Transferência G(s):
    2.0⋅s + 3.0     
────────────────────
     2              
1.0⋅s  + 5.0⋅s + 6.0

Escolha o formato de realização:
1 - Forma Canônica Controlável (FCC)
2 - Forma Canônica Observável (FCO)

REALIZAÇÃO: Forma Canônica Controlável (FCC)

Matriz A:
[[ 0.  1.]
 [-6. -5.]]

Matriz B:
[[0.]
 [1.]]

Matriz C:
[[3. 2.]]

Matriz D:
[[0.]]

ANÁLISE DE PROPRIEDADES DO ESPAÇO DE ESTADOS
Ordem do sistema (n): 2
Posto da Matriz de Controlabilidade: 2 -> CONTROLÁVEL
Posto da Matriz de Observabilidade:  2 -> OBSERVÁVEL

Autovalores de A (Pólos do Sistema):
  λ1 = -2.0000+0.0000j
  λ2 = -3.0000+0.0000j
